# Complementos de Investigação Operacional — capítulo 8: Decisão multicritério (AHP)
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/08_ahp.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 8 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 8.1 Decisão multicritério: o AHP, comparações par a par e consistência

Exemplo [`ex08_1_ahp.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_1_ahp/ex08_1_ahp.py) · funções da UC: [`ahp.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_1_ahp/ahp.py)

```text
Reproduz os exemplos do deck 8.1 (o AHP, comparações par a par e consistência).

Carrinhas elétricas: critérios custo, autonomia, carga e assistência, juízos 2, 3, 5, 2, 3, 2
=> pesos 0.482, 0.272, 0.158, 0.088; lambda_max = 4.015, IC = 0.0048, RC = 0.005 (coerente);
vetor próprio 0.4829, 0.2720, 0.1570, 0.0882.
Slide «Quando os juízos não batem certo»: com a13 = 1/2, RC = 0.173; o par custo–carga é o mais
afastado (0.40 e 2.49); com a13 = 3, RC = 0.005; com a13 = 1, RC = 0.069.
Assistência (A, B, C): 0.261, 0.633, 0.106, RC = 0.033.
Para resolver na aula: Exemplo 1 (frutos: 0.283, 0.643, 0.074, RC = 0.056) e
Exemplo 2 (emprego: pesos 0.094, 0.738, 0.168, RC = 0.012; A com 0.735).
```

In [2]:
import numpy as np

from ahp import matriz, prioridades


def mat2str(A, casas=4):
    """Matriz/vetor arredondado, no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.round(np.asarray(A, float), casas) + 0.0)
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def perto(x, v, tol):
    """x e v iguais a menos de tol (em todas as componentes)."""
    return bool(np.all(np.abs(np.asarray(x, float) - np.asarray(v, float)) <= tol))


def passo_a_passo(A):
    """Imprime as contas do método das colunas e devolve (w, lambda_max, IC, RC)."""
    A = np.asarray(A, float)
    w, lam, IC, RC = prioridades(A)
    print("  somas das colunas: %s" % mat2str(A.sum(0)))
    print("  matriz normalizada: %s" % mat2str(A / A.sum(0)))
    print("  pesos w: %s" % mat2str(w))
    print("  Aw: %s" % mat2str(A @ w))
    print("  (Aw)_i/w_i: %s" % mat2str(A @ w / w))
    print("  lambda_max = %.4f  IC = %.4f  RC = %.4f -> %s"
          % (lam, IC, RC, "coerente" if RC < 0.1 else "incoerente"))
    return w, lam, IC, RC


ok = True
print("Complementos de IO — deck 8.1: o AHP, comparações par a par e consistência")

# ------------------------------------------------------------ carrinhas: pesos dos critérios
crit = ["custo", "autonomia", "carga", "assistência"]
C = matriz(4, {(0, 1): 2, (0, 2): 3, (0, 3): 5, (1, 2): 2, (1, 3): 3, (2, 3): 2})
print("\nCarrinhas: critérios custo, autonomia, carga, assistência")
w, lam, IC, RC = passo_a_passo(C)
wv, lamv, _, _ = prioridades(C, "vetor")
print("  vetor próprio: w = %s  lambda_max = %.4f" % (mat2str(wv), lamv))
c = [perto(C.sum(0), [2.033, 3.833, 6.5, 11], 5e-4),
     perto(C / C.sum(0), [[0.492, 0.522, 0.462, 0.455], [0.246, 0.261, 0.308, 0.273],
     [0.164, 0.130, 0.154, 0.182], [0.098, 0.087, 0.077, 0.091]], 5e-4),
     perto(w, [0.4824, 0.2718, 0.1575, 0.0883], 5e-5),
     perto(C @ w, [1.940, 1.093, 0.631, 0.354], 5e-4),
     perto(C @ w / w, [4.021, 4.021, 4.005, 4.011], 5e-4),
     perto(lam, 4.0145, 5e-5),
     perto(IC, 0.0048, 5e-5),
     perto(RC, 0.005, 5e-4),
     perto(wv, [0.4829, 0.2720, 0.1570, 0.0882], 5e-5),
     perto(lamv, 4.0145, 5e-5)]
ok &= all(c)

# ------------------------------------------------------------ juízo incoerente a13 = 1/2
Cm = C.copy()
Cm[0, 2], Cm[2, 0] = 1 / 2, 2
wm, lamm, _, RCm = prioridades(Cm)
R = Cm * wm[None, :] / wm[:, None]             # a_ij w_j / w_i (devia ser perto de 1)
pior, ip, jp = 0, 0, 0
for i in range(4):
    for j in range(i + 1, 4):
        if abs(np.log(R[i, j])) > pior:
            pior, ip, jp = abs(np.log(R[i, j])), i, j
print("\nJuízo incoerente a13 = 1/2 (os outros cinco iguais)")
print("  lambda_max = %.4f  RC = %.4f -> %s" % (lamm, RCm, "coerente" if RCm < 0.1 else "incoerente"))
print("  pesos w: %s" % mat2str(wm))
print("  a_ij w_j / w_i: %s" % mat2str(R, 2))
print("  par mais afastado de 1: %s–%s (%.2f e %.2f)" % (crit[ip], crit[jp], R[ip, jp], R[jp, ip]))
RCa = {}
for a in (3, 1):
    Ct = C.copy()
    Ct[0, 2], Ct[2, 0] = a, 1 / a
    RCa[a] = prioridades(Ct)[3]
print("  a13 = 3: RC = %.4f;  a13 = 1: RC = %.4f" % (RCa[3], RCa[1]))
c = [perto(lamm, 4.467, 5e-4),
     perto(RCm, 0.173, 5e-4),
     perto(wm, [0.343, 0.292, 0.276, 0.089], 5e-4),
     perto(R, [[1, 1.70, 0.40, 1.30], [0.59, 1, 1.89, 0.92], [2.49, 0.53, 1, 0.65],
           [0.77, 1.09, 1.54, 1]], 5e-3),
     (ip, jp) == (0, 2),
     perto(RCa[3], 0.005, 5e-4),
     perto(RCa[1], 0.069, 5e-4)]
ok &= all(c)

# ------------------------------------------------------------ assistência (critério qualitativo)
S = matriz(3, {(0, 1): 1 / 3, (0, 2): 3, (1, 2): 5})
ws, lams, ICs, RCs = prioridades(S)
print("\nAssistência (A, B, C): prioridades %s  lambda_max = %.4f  IC = %.4f  RC = %.4f"
      % (mat2str(ws), lams, ICs, RCs))
c = [perto(ws, [0.2605, 0.6333, 0.1062], 5e-5),
     perto(lams, 3.039, 5e-4),
     perto(ICs, 0.019, 5e-4),
     perto(RCs, 0.033, 5e-4)]
ok &= all(c)

# ------------------------------------------------------------ Exemplo 1: três frutos
F = matriz(3, {(0, 1): 1 / 3, (0, 2): 5, (1, 2): 7})   # F2 face a F1: 3; F1 face a F3: 5; F2 face a F3: 7
print("\nExemplo 1 (frutos F1, F2, F3)")
wf, lamf, ICf, RCf = passo_a_passo(F)
ordem = np.argsort(-wf)
print("  ordem: %s;  coerência perfeita pediria F2/F3 = 3 x 5 = 15 (o juízo foi 7)"
      % ", ".join("F%d" % (i + 1) for i in ordem))
c = [perto(F.sum(0), [4.2, 1.476, 13], 5e-4),
     perto(wf, [0.283, 0.643, 0.074], 5e-4),
     perto(F @ wf / wf, [3.062, 3.121, 3.013], 5e-4),
     perto(lamf, 3.066, 5e-4),
     perto(ICf, 0.033, 5e-4),
     perto(RCf, 0.056, 5e-4),
     list(ordem) == [1, 0, 2]]
ok &= all(c)

# ------------------------------------------------------------ Exemplo 2: duas propostas de emprego
E = [[1, 1 / 7, 1 / 2], [7, 1, 5], [2, 1 / 5, 1]]
print("\nExemplo 2 (emprego, propostas A e B)")
we, lame, ICe, RCe = passo_a_passo(E)
# salário como comparação coerente (B paga 2500/2000 = 1.25 vezes mais); critérios 2 e 3: A 3 e 5 vezes
Me = np.column_stack([prioridades(matriz(2, {(0, 1): 2000 / 2500}))[0],
                      prioridades(matriz(2, {(0, 1): 3}))[0],
                      prioridades(matriz(2, {(0, 1): 5}))[0]])
Se = Me @ we
print("  prioridades locais (linhas A, B; colunas salário, critério 2, critério 3): %s" % mat2str(Me))
print("  pontuação: A %.4f  B %.4f -> proposta %s" % (Se[0], Se[1], "AB"[int(np.argmax(Se))]))
print("  implícito c2/c3 = 7 x 1/2 = %.1f; o juízo foi 5" % (E[1][0] * E[0][2]))
c = [perto(np.sum(E, 0), [10, 1.343, 6.5], 5e-4),
     perto(we, [0.094, 0.738, 0.168], 5e-4),
     perto(lame, 3.014, 5e-4),
     perto(ICe, 0.007, 5e-4),
     perto(RCe, 0.012, 5e-4),
     perto(Me, [[0.444, 0.75, 0.833], [0.556, 0.25, 0.167]], 5e-4),
     perto(Se, [0.735, 0.265], 5e-4),
     np.argmax(Se) == 0]
ok &= all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 8.1: o AHP, comparações par a par e consistência

Carrinhas: critérios custo, autonomia, carga, assistência
  somas das colunas: [2.0333 3.8333 6.5 11]
  matriz normalizada: [0.4918 0.5217 0.4615 0.4545;0.2459 0.2609 0.3077 0.2727;0.1639 0.1304 0.1538 0.1818;0.0984 0.087 0.0769 0.0909]
  pesos w: [0.4824 0.2718 0.1575 0.0883]
  Aw: [1.94 1.0929 0.6308 0.3541]
  (Aw)_i/w_i: [4.0214 4.0209 4.0048 4.011]
  lambda_max = 4.0145  IC = 0.0048  RC = 0.0054 -> coerente
  vetor próprio: w = [0.4829 0.272 0.157 0.0882]  lambda_max = 4.0145

Juízo incoerente a13 = 1/2 (os outros cinco iguais)
  lambda_max = 4.4668  RC = 0.1729 -> incoerente
  pesos w: [0.3429 0.2922 0.2757 0.0892]
  a_ij w_j / w_i: [1 1.7 0.4 1.3;0.59 1 1.89 0.92;2.49 0.53 1 0.65;0.77 1.09 1.54 1]
  par mais afastado de 1: custo–carga (0.40 e 2.49)
  a13 = 3: RC = 0.0054;  a13 = 1: RC = 0.0686

Assistência (A, B, C): prioridades [0.2605 0.6333 0.1062]  lambda_max = 3.0387  IC = 0.0194  RC = 0.0334

Exempl

## 8.2 Decisão multicritério: atributos quantitativos e agregação

Exemplo [`ex08_2_attributes.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_2_attributes/ex08_2_attributes.py) · funções da UC: [`ahp.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_1_ahp/ahp.py), [`atributos.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_2_attributes/atributos.py)

```text
Reproduz os exemplos do deck 8.2 (atributos quantitativos e agregação).

Carrinhas: custo indireto (38 000, 44 000, 33 000 €), autonomia direta (220, 300, 160 km), carga direta
(900, 1000, 750 kg) e assistência pelas comparações de 8.1 => prioridades locais 0.332, 0.286, 0.382 no custo, ...;
pontuações A 0.324, B 0.373, C 0.302 -> B. C ganha 0.046 no custo; B recupera 0.056 na autonomia e 0.047
na assistência. A normalização comprime: 0.382 / 0.286 = 44 000 / 33 000 = 1.33.
Filtro «autonomia >= 200 km» (sem C): A 0.471, B 0.529 -> B.
Para resolver na aula: fornecedores (custo da MP, transporte e tempo indiretos; rendimento direto)
Exemplo 3 (exame 11/07/2019): RC = 0.062, F3 com 0.3034, margem 0.035 para F2;
Exemplo 4 (2.º teste 01/06/2018): RC = 0.027, F3 com 0.344, margem 0.108 para F4;
Exemplo 5 (2.º teste 27/06/2019): RC = 0.012, F3 com 0.2649, margem só 0.014 para F1;
no Exemplo 5, os erros frequentes (régua trocada, transporte como direto, soma sem pesos) dão F2.
```

In [3]:
import numpy as np

from ahp import matriz, prioridades
from atributos import direto, indireto, agrega


def mat2str(A, casas=4):
    """Matriz/vetor arredondado, no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.round(np.asarray(A, float), casas) + 0.0)
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def perto(x, v, tol):
    """x e v iguais a menos de tol (em todas as componentes)."""
    return bool(np.all(np.abs(np.asarray(x, float) - np.asarray(v, float)) <= tol))


ok = True
print("Complementos de IO — deck 8.2: atributos quantitativos e agregação")

# ------------------------------------------------------------ carrinhas
C = matriz(4, {(0, 1): 2, (0, 2): 3, (0, 3): 5, (1, 2): 2, (1, 3): 3, (2, 3): 2})   # de 8.1
S = matriz(3, {(0, 1): 1 / 3, (0, 2): 3, (1, 2): 5})                              # assistência, de 8.1
w = prioridades(C)[0]
custo, autonomia, carga = [38000, 44000, 33000], [220, 300, 160], [900, 1000, 750]
M = np.column_stack([indireto(custo), direto(autonomia), direto(carga), prioridades(S)[0]])
contrib = M * w
s = agrega(M, w)
print("\nCarrinhas A, B, C (colunas: custo ind., autonomia dir., carga dir., assistência)")
print("  custo: 10^5/v = %s (soma %.4f)" % (mat2str(1e5 / np.array(custo)), np.sum(1e5 / np.array(custo))))
print("  prioridades locais: %s" % mat2str(M))
print("  pesos w (8.1): %s" % mat2str(w))
print("  contribuições w_k p_ik: %s" % mat2str(contrib))
print("  pontuação S: %s -> carrinha %s" % (mat2str(s), "ABC"[int(np.argmax(s))]))
ganho_custo = contrib[2, 0] - contrib[1, 0]
ganho_aut = contrib[1, 1] - contrib[2, 1]
ganho_ass = contrib[1, 3] - contrib[2, 3]
print("  C ganha %.4f no custo; B recupera %.4f na autonomia e %.4f na assistência"
      % (ganho_custo, ganho_aut, ganho_ass))
print("  compressão: B custa mais %.0f%% do que C; prioridades no custo C/B = %.4f = 44000/33000"
      % (100 * (custo[1] / custo[2] - 1), M[2, 0] / M[1, 0]))
Mf = np.column_stack([indireto(custo[:2]), direto(autonomia[:2]), direto(carga[:2]),
                      prioridades(matriz(2, {(0, 1): 1 / 3}))[0]])
sf = agrega(Mf, w)
print("  filtro autonomia >= 200 km (sem C): A %.4f  B %.4f -> %s" % (sf[0], sf[1], "AB"[int(np.argmax(sf))]))
c = [perto(1e5 / np.array(custo), [2.632, 2.273, 3.030], 5e-4),
     perto(np.sum(1e5 / np.array(custo)), 7.935, 5e-4),
     perto(M, [[0.3317, 0.3235, 0.3396, 0.2605], [0.2864, 0.4412, 0.3774, 0.6333],
               [0.3819, 0.2353, 0.2830, 0.1062]], 5e-5),
     perto(contrib, [[0.1600, 0.0879, 0.0535, 0.0230], [0.1382, 0.1199, 0.0594, 0.0559],
                     [0.1842, 0.0640, 0.0446, 0.0094]], 5e-5),
     perto(s, [0.324, 0.373, 0.302], 5e-4),
     np.argmax(s) == 1,
     perto([ganho_custo, ganho_aut, ganho_ass], [0.046, 0.056, 0.047], 5e-4),
     perto(M[2, 0] / M[1, 0], 44000 / 33000, 1e-12),
     perto(sf, [0.471, 0.529], 5e-4)]
ok &= all(c)

# ------------------------------------------------------------ Exemplos 3, 4 e 5: fornecedores
# (nome, juízos C1..C4, custo MP, transporte, tempo, rendimento) e o esperado (notas do docente):
# (w, lambda_max, RC, S, índice do escolhido, margem, índice do 2.º)
casos = [
    ("Exemplo 3 (exame, 11/07/2019)",
     {(0, 1): 7, (0, 2): 5, (0, 3): 1 / 3, (1, 2): 1 / 3, (1, 3): 1 / 9, (2, 3): 1 / 7},
     [12000, 6000, 18000, 12000], [400, 600, 200, 800], [15, 10, 20, 5], [2, 1.5, 3, 1],
     ([0.2913, 0.0445, 0.0903, 0.5739], 4.168, 0.062, [0.2406, 0.2684, 0.3034, 0.1876], 2, 0.035, 1)),
    ("Exemplo 4 (2.º teste, 01/06/2018)",
     {(0, 1): 1 / 5, (0, 2): 3, (0, 3): 1, (1, 2): 7, (1, 3): 5, (2, 3): 1 / 3},
     [10000, 6000, 16000, 11000], [400, 700, 200, 300], [12, 10, 20, 15], [2.5, 2, 3, 1],
     ([0.1542, 0.6279, 0.0637, 0.1542], 4.074, 0.027, [0.2278, 0.1918, 0.3440, 0.2364], 2, 0.108, 3)),
    ("Exemplo 5 (2.º teste, 27/06/2019)",
     {(0, 1): 1 / 3, (0, 2): 5, (0, 3): 1, (1, 2): 9, (1, 3): 3, (2, 3): 1 / 5},
     [11000, 7000, 15000, 12000], [1400, 1700, 1200, 1300], [15, 13, 23, 18], [3, 2.5, 3.5, 2],
     ([0.2055, 0.5416, 0.0474, 0.2055], 4.033, 0.012, [0.2509, 0.2479, 0.2649, 0.2362], 2, 0.014, 0)),
]
for nome, juizos, mp, tr, te, re, esp in casos:
    Cf = matriz(4, juizos)
    wf, lamf, ICf, RCf = prioridades(Cf)
    Mf = np.column_stack([indireto(mp), indireto(tr), indireto(te), direto(re)])
    sf = agrega(Mf, wf)
    o = np.argsort(-sf)
    print("\n%s" % nome)
    print("  matriz dos critérios: %s" % mat2str(Cf))
    print("  somas das colunas: %s" % mat2str(Cf.sum(0)))
    print("  pesos w: %s" % mat2str(wf))
    print("  (Aw)_i/w_i: %s" % mat2str(Cf @ wf / wf))
    print("  lambda_max = %.4f  IC = %.4f  RC = %.4f -> %s"
          % (lamf, ICf, RCf, "coerente" if RCf < 0.1 else "incoerente"))
    print("  prioridades locais (linhas F1..F4; colunas MP, transp., tempo, rend.): %s" % mat2str(Mf))
    print("  pontuação S: %s -> F%d, margem %.4f para F%d"
          % (mat2str(sf), o[0] + 1, sf[o[0]] - sf[o[1]], o[1] + 1))
    w_e, lam_e, rc_e, s_e, b_e, m_e, seg_e = esp
    c = [perto(wf, w_e, 5e-5),
         perto(lamf, lam_e, 5e-4),
         perto(RCf, rc_e, 5e-4),
         perto(sf, s_e, 5e-5),
         o[0] == b_e,
         perto(sf[o[0]] - sf[o[1]], m_e, 5e-4),
         o[1] == seg_e]
    ok &= all(c)

# ------------------------------------------------------------ Exemplo 5: os erros frequentes mudam a escolha
nome, juizos, mp, tr, te, re, _ = casos[2]
Cf = matriz(4, juizos)
wf = prioridades(Cf)[0]
Mf = np.column_stack([indireto(mp), indireto(tr), indireto(te), direto(re)])
s_regua = agrega(Mf, prioridades(Cf.T)[0])                    # régua trocada: matriz transposta
Md = Mf.copy()
Md[:, 1] = direto(tr)
s_dir = agrega(Md, wf)                                        # transporte tratado como direto
s_soma = Mf.sum(1)                                            # somar sem ponderar
print("\nExemplo 5, erros frequentes:")
for rot, v in (("régua trocada", s_regua), ("transporte como direto", s_dir), ("soma sem pesos", s_soma)):
    print("  %s: S = %s -> F%d (%.4f)" % (rot, mat2str(v), np.argmax(v) + 1, v.max()))
c = [np.argmax(s_regua) == 1,
     perto(s_regua.max(), 0.305, 5e-4),
     np.argmax(s_dir) == 1,
     perto(s_dir.max(), 0.303, 5e-4),
     np.argmax(s_soma) == 1,
     perto(s_soma.max(), 1.119, 5e-4)]
ok &= all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 8.2: atributos quantitativos e agregação

Carrinhas A, B, C (colunas: custo ind., autonomia dir., carga dir., assistência)
  custo: 10^5/v = [2.6316 2.2727 3.0303] (soma 7.9346)
  prioridades locais: [0.3317 0.3235 0.3396 0.2605;0.2864 0.4412 0.3774 0.6333;0.3819 0.2353 0.283 0.1062]
  pesos w (8.1): [0.4824 0.2718 0.1575 0.0883]
  contribuições w_k p_ik: [0.16 0.0879 0.0535 0.023;0.1382 0.1199 0.0594 0.0559;0.1842 0.064 0.0446 0.0094]
  pontuação S: [0.3244 0.3734 0.3021] -> carrinha B
  C ganha 0.0461 no custo; B recupera 0.0560 na autonomia e 0.0465 na assistência
  compressão: B custa mais 33% do que C; prioridades no custo C/B = 1.3333 = 44000/33000
  filtro autonomia >= 200 km (sem C): A 0.4705  B 0.5295 -> B

Exemplo 3 (exame, 11/07/2019)
  matriz dos critérios: [1 7 5 0.3333;0.1429 1 0.3333 0.1111;0.2 3 1 0.1429;3 9 7 1]
  somas das colunas: [4.3429 20 13.3333 1.5873]
  pesos w: [0.2913 0.0445 0.0903 0.5739]
  (Aw)_i/w_i: [4.2746 4.0462 4.032 4.321]
  

## 8.3 Decisão multicritério: análise de sensibilidade

Exemplo [`ex08_3_sensitivity.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_3_sensitivity/ex08_3_sensitivity.py) · funções da UC: [`ahp.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_1_ahp/ahp.py), [`atributos.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_2_attributes/atributos.py), [`sensibilidade.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/08_ahp/08_3_sensitivity/sensibilidade.py)

```text
Reproduz os exemplos do deck 8.3 (análise de sensibilidade).

Carrinhas (8.1 e 8.2): fazendo variar o peso do custo, S(0) = (0.318, 0.455, 0.228) e S(1) = (0.332, 0.286, 0.382);
C passa A a p = 0.641 e passa B a p = 0.704 (d0 = 0.2267, d1 = -0.0955); com a autonomia, a carga ou a
assistência B fica em 1.º para qualquer peso.
Um juízo (a12 = 1, 2, 5, 9): B ganha sempre; com custo 7/9/9 face aos outros, C empata com B (0.3364 e 0.3365).
Todos os juízos (20 000 repetições, cada juízo sobe/desce um degrau ou fica): B em 100%; RC > 0.1 em 1.1%.
Para resolver na aula: Exemplo 6 (= Exemplo 5): margem 0.014 para F1; F2 passa F3 com o custo da MP acima
de 0.268 e com o transporte abaixo de 0.426; F3 em 89.9% das simulações.
Exemplo 7 (= Exemplo 3): F3 enquanto o rendimento pesar pelo menos 0.484; F3 em 100% das simulações.
As simulações usam semente fixa; a versão MATLAB usa outro gerador e dá frequências próximas, não iguais.
```

In [4]:
import numpy as np

from ahp import matriz, prioridades
from atributos import direto, indireto, agrega
from sensibilidade import pesos_com, sensibilidade, viragem, simula_juizos


def mat2str(A, casas=4):
    """Matriz/vetor arredondado, no formato do mat2str do MATLAB (para as saídas serem iguais)."""
    A = np.atleast_2d(np.round(np.asarray(A, float), casas) + 0.0)
    return "[" + ";".join(" ".join("%.15g" % x for x in linha) for linha in A) + "]"


def perto(x, v, tol):
    """x e v iguais a menos de tol (em todas as componentes)."""
    return bool(np.all(np.abs(np.asarray(x, float) - np.asarray(v, float)) <= tol))


def retas(M, w, k, nomes, ncrit):
    """Imprime S(0), S(1), as viragens do vencedor atual e o intervalo de p em que ele se mantém.
    Devolve (limite inferior, quem passa abaixo, limite superior, quem passa acima, {j: viragem})."""
    M = np.asarray(M, float)
    b = int(np.argmax(agrega(M, w)))
    s0 = agrega(M, pesos_com(w, k, 0))
    print("  %s (peso atual %.4f): S(0) = %s  S(1) = %s" % (ncrit, w[k], mat2str(s0), mat2str(M[:, k])))
    lo, qlo, hi, qhi, vir = 0.0, -1, 1.0, -1, {}
    for j in range(len(M)):
        if j == b:
            continue
        p = viragem(M, w, k, b, j)
        if p is None:
            continue
        vir[j] = p
        print("    %s–%s: d0 = %.4f  d1 = %.4f  p* = %.4f"
              % (nomes[b], nomes[j], s0[b] - s0[j], M[b, k] - M[j, k], p))
        if w[k] < p < hi:
            hi, qhi = p, j
        if lo < p < w[k]:
            lo, qlo = p, j
    if qlo < 0 and qhi < 0:
        print("    %s em 1.º para qualquer p em [0, 1]" % nomes[b])
    else:
        txt = "    %s em 1.º para p em [%.4f, %.4f]" % (nomes[b], lo, hi)
        if qlo >= 0:
            txt += "; abaixo passa %s" % nomes[qlo]
        if qhi >= 0:
            txt += "; acima passa %s" % nomes[qhi]
        print(txt)
    return lo, qlo, hi, qhi, vir


ok = True
print("Complementos de IO — deck 8.3: análise de sensibilidade")

# ------------------------------------------------------------ carrinhas: fazer variar um peso
C = matriz(4, {(0, 1): 2, (0, 2): 3, (0, 3): 5, (1, 2): 2, (1, 3): 3, (2, 3): 2})
S = matriz(3, {(0, 1): 1 / 3, (0, 2): 3, (1, 2): 5})
w = prioridades(C)[0]
M = np.column_stack([indireto([38000, 44000, 33000]), direto([220, 300, 160]), direto([900, 1000, 750]),
                     prioridades(S)[0]])
crit = ["custo", "autonomia", "carga", "assistência"]
print("\nCarrinhas: pontuação S = %s; pesos w = %s" % (mat2str(agrega(M, w)), mat2str(w)))
res = [retas(M, w, k, "ABC", crit[k]) for k in range(4)]
pBC, pCA = viragem(M, w, 0, 1, 2), viragem(M, w, 0, 2, 0)
print("  no custo: C passa A a p = %.4f (S = %s) e passa B a p = %.4f (S = %s)"
      % (pCA, mat2str(agrega(M, pesos_com(w, 0, pCA))), pBC, mat2str(agrega(M, pesos_com(w, 0, pBC)))))
grelha = np.linspace(0, 1, 11)
T = sensibilidade(M, w, 0, grelha)
print("  sensibilidade no peso do custo (o gráfico do slide):")
for p, linha in zip(grelha, T):
    print("    p = %.1f: A %.4f  B %.4f  C %.4f" % (p, linha[0], linha[1], linha[2]))
gfina = np.linspace(0, 1, 1001)
Bgrelha = gfina[sensibilidade(M, w, 0, gfina).argmax(1) == 1]
c = [perto(agrega(M, pesos_com(w, 0, 0)), [0.3177, 0.4545, 0.2278], 5e-5),
     perto(M[:, 0], [0.3317, 0.2864, 0.3819], 5e-5),
     perto(pBC, 0.704, 5e-4),
     perto(pCA, 0.641, 5e-4),
     res[0][2:4] == (pBC, 2) and res[0][1] == -1,
     all(r[1] == -1 and r[3] == -1 for r in res[1:]),
     perto(agrega(M, pesos_com(w, 0, pBC))[1:], [0.336, 0.336], 5e-4),
     perto(agrega(M, pesos_com(w, 0, pCA))[[0, 2]], [0.3266, 0.3266], 5e-5),
     perto(T[[0, -1]], [[0.3177, 0.4545, 0.2278], [0.3317, 0.2864, 0.3819]], 5e-5),
     Bgrelha.min() == 0 and perto(Bgrelha.max(), 0.703, 1e-9)]
ok &= all(c)

# ------------------------------------------------------------ um juízo: custo face a autonomia
print("\nUm juízo: a12 (custo face a autonomia; era 2)")
tab = []
for a in (1, 2, 5, 9):
    Ct = C.copy()
    Ct[0, 1], Ct[1, 0] = a, 1 / a
    wt, _, _, rct = prioridades(Ct)
    st = agrega(M, wt)
    tab.append([a, wt[0], rct, st[0], st[1], st[2]])
    print("  a12 = %d: w_custo = %.4f  RC = %.4f  S = %s -> %s" % (a, wt[0], rct, mat2str(st), "ABC"[int(np.argmax(st))]))
C7 = matriz(4, {(0, 1): 7, (0, 2): 9, (0, 3): 9, (1, 2): 2, (1, 3): 3, (2, 3): 2})
s7 = agrega(M, prioridades(C7)[0])
print("  custo 7, 9 e 9 vezes mais importante do que autonomia, carga e assistência: S = %s" % mat2str(s7))
tab = np.array(tab)
c = [perto(tab[:, 1:], [[0.4159, 0.0126, 0.3238, 0.3840, 0.2922], [0.4824, 0.0054, 0.3244, 0.3734, 0.3021],
                        [0.5562, 0.0665, 0.3252, 0.3615, 0.3133], [0.5887, 0.1583, 0.3256, 0.3561, 0.3182]], 5e-5),
     np.all(tab[:, 4] > tab[:, [3, 5]].max(1)),
     perto(s7[1:], [0.3364, 0.3365], 5e-5)]
ok &= all(c)

# ------------------------------------------------------------ todos os juízos: simulação
vit, inc = simula_juizos(C, M, {3: S})
print("\nTodos os juízos (20000 repetições, ±1 degrau na escala de Saaty, semente 1):")
print("  1.º lugar: A %.1f%%  B %.1f%%  C %.1f%%;  RC > 0.1 em %.1f%% das matrizes"
      % (100 * vit[0], 100 * vit[1], 100 * vit[2], 100 * inc))
c = [vit[1] >= 0.995, perto(inc, 0.011, 0.004)]       # frequências: tolerância de simulação
ok &= all(c)

# ------------------------------------------------------------ Exemplos 6 e 7: fornecedores
F4 = ["F1", "F2", "F3", "F4"]
critf = ["custo MP", "transporte", "tempo", "rendimento"]
J5 = {(0, 1): 1 / 3, (0, 2): 5, (0, 3): 1, (1, 2): 9, (1, 3): 3, (2, 3): 1 / 5}         # Exemplo 5 de 8.2
C5 = matriz(4, J5)
w5 = prioridades(C5)[0]
M5 = np.column_stack([indireto([11000, 7000, 15000, 12000]), indireto([1400, 1700, 1200, 1300]),
                      indireto([15, 13, 23, 18]), direto([3, 2.5, 3.5, 2])])
s5 = agrega(M5, w5)
o5 = np.argsort(-s5)
print("\nExemplo 6 (= Exemplo 5; 2.º teste, 27/06/2019): S = %s" % mat2str(s5))
print("  a) %s em 1.º, margem %.4f para %s" % (F4[o5[0]], s5[o5[0]] - s5[o5[1]], F4[o5[1]]))
r6 = [retas(M5, w5, k, F4, critf[k]) for k in range(3)]
vit6, _ = simula_juizos(C5, M5)
print("  simulação (±1 degrau): F3 em 1.º em %.1f%% das repetições" % (100 * vit6[2]))
c = [o5[0] == 2 and o5[1] == 0,
     perto(s5[2] - s5[0], 0.014, 5e-4),
     perto(agrega(M5, pesos_com(w5, 0, 0)), [0.2545, 0.2157, 0.2885, 0.2412], 5e-5),
     perto([r6[0][4][j] for j in (1, 0, 3)], [0.268, 0.350, 0.522], 5e-4),
     r6[0][2:4] == (r6[0][4][1], 1),
     perto(agrega(M5, pesos_com(w5, 1, 0)), [0.2569, 0.3015, 0.2391, 0.2025], 5e-5),
     perto([r6[1][4][j] for j in (1, 0)], [0.426, 0.303], 5e-4),
     r6[1][:2] == (r6[1][4][1], 1),
     perto(r6[2][2], 0.152, 5e-4) and r6[2][3] == 1,
     perto(vit6[2], 0.899, 0.01)]
ok &= all(c)

J3 = {(0, 1): 7, (0, 2): 5, (0, 3): 1 / 3, (1, 2): 1 / 3, (1, 3): 1 / 9, (2, 3): 1 / 7}  # Exemplo 3 de 8.2
C3 = matriz(4, J3)
w3 = prioridades(C3)[0]
M3 = np.column_stack([indireto([12000, 6000, 18000, 12000]), indireto([400, 600, 200, 800]),
                      indireto([15, 10, 20, 5]), direto([2, 1.5, 3, 1])])
s3 = agrega(M3, w3)
o3 = np.argsort(-s3)
print("\nExemplo 7 (= Exemplo 3; exame, 11/07/2019): S = %s; %s em 1.º, margem %.4f para %s"
      % (mat2str(s3), F4[o3[0]], s3[o3[0]] - s3[o3[1]], F4[o3[1]]))
r7 = [retas(M3, w3, k, F4, critf[k]) for k in (3, 0)]
vit7, _ = simula_juizos(C3, M3)
print("  simulação (±1 degrau): F3 em 1.º em %.1f%% das repetições" % (100 * vit7[2]))
c = [o3[0] == 2 and perto(s3[2] - s3[1], 0.035, 5e-4),
     perto(agrega(M3, pesos_com(w3, 3, 0)), [0.2055, 0.3606, 0.1732, 0.2607], 5e-5),
     perto([r7[0][4][j] for j in (1, 3, 0)], [0.484, 0.247, 0.195], 5e-4),
     r7[0][:2] == (r7[0][4][1], 1) and r7[0][3] == -1,
     perto(r7[1][2], 0.369, 5e-4) and r7[1][3] == 1,
     vit7[2] >= 0.995]
ok &= all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 8.3: análise de sensibilidade

Carrinhas: pontuação S = [0.3244 0.3734 0.3021]; pesos w = [0.4824 0.2718 0.1575 0.0883]
  custo (peso atual 0.4824): S(0) = [0.3177 0.4545 0.2278]  S(1) = [0.3317 0.2864 0.3819]
    B–A: d0 = 0.1369  d1 = -0.0452  p* = 0.7516
    B–C: d0 = 0.2267  d1 = -0.0955  p* = 0.7037
    B em 1.º para p em [0.0000, 0.7037]; acima passa C
  autonomia (peso atual 0.2718): S(0) = [0.3248 0.3482 0.3271]  S(1) = [0.3235 0.4412 0.2353]
    B em 1.º para qualquer p em [0, 1]
  carga (peso atual 0.1575): S(0) = [0.3216 0.3727 0.3057]  S(1) = [0.3396 0.3774 0.283]
    B em 1.º para qualquer p em [0, 1]
  assistência (peso atual 0.0883): S(0) = [0.3306 0.3483 0.3211]  S(1) = [0.2605 0.6333 0.1062]
    B em 1.º para qualquer p em [0, 1]
  no custo: C passa A a p = 0.6414 (S = [0.3266 0.3467 0.3266]) e passa B a p = 0.7037 (S = [0.3275 0.3362 0.3362])
  sensibilidade no peso do custo (o gráfico do slide):
    p = 0.0: A 0.3177  B 0.4545  C 0.2278
    


Todos os juízos (20000 repetições, ±1 degrau na escala de Saaty, semente 1):
  1.º lugar: A 0.0%  B 100.0%  C 0.0%;  RC > 0.1 em 1.1% das matrizes

Exemplo 6 (= Exemplo 5; 2.º teste, 27/06/2019): S = [0.2509 0.2479 0.2649 0.2362]
  a) F3 em 1.º, margem 0.0140 para F1
  custo MP (peso atual 0.2055): S(0) = [0.2545 0.2157 0.2885 0.2412]  S(1) = [0.2369 0.3723 0.1737 0.2171]
    F3–F1: d0 = 0.0340  d1 = -0.0632  p* = 0.3499
    F3–F2: d0 = 0.0728  d1 = -0.1985  p* = 0.2683
    F3–F4: d0 = 0.0474  d1 = -0.0434  p* = 0.5216
    F3 em 1.º para p em [0.0000, 0.2683]; acima passa F2
  transporte (peso atual 0.5416): S(0) = [0.2569 0.3015 0.2391 0.2025]  S(1) = [0.2459 0.2025 0.2869 0.2648]
    F3–F1: d0 = -0.0178  d1 = 0.0410  p* = 0.3030
    F3–F2: d0 = -0.0625  d1 = 0.0844  p* = 0.4255
    F3 em 1.º para p em [0.4255, 1.0000]; abaixo passa F2
  tempo (peso atual 0.0474): S(0) = [0.2497 0.2445 0.2692 0.2366]  S(1) = [0.2748 0.317 0.1792 0.229]
    F3–F1: d0 = 0.0195  d1 = -0.0956  p* = 0.169

  simulação (±1 degrau): F3 em 1.º em 89.9% das repetições

Exemplo 7 (= Exemplo 3; exame, 11/07/2019): S = [0.2406 0.2684 0.3034 0.1876]; F3 em 1.º, margem 0.0350 para F2
  rendimento (peso atual 0.5739): S(0) = [0.2055 0.3606 0.1732 0.2607]  S(1) = [0.2667 0.2 0.4 0.1333]
    F3–F1: d0 = -0.0323  d1 = 0.1333  p* = 0.1948
    F3–F2: d0 = -0.1874  d1 = 0.2000  p* = 0.4837
    F3–F4: d0 = -0.0875  d1 = 0.2667  p* = 0.2471
    F3 em 1.º para p em [0.4837, 1.0000]; abaixo passa F2
  custo MP (peso atual 0.2913): S(0) = [0.2514 0.2026 0.3694 0.1767]  S(1) = [0.2143 0.4286 0.1429 0.2143]
    F3–F1: d0 = 0.1180  d1 = -0.0714  p* = 0.6228
    F3–F2: d0 = 0.1668  d1 = -0.2857  p* = 0.3686
    F3–F4: d0 = 0.1927  d1 = -0.0714  p* = 0.7296
    F3 em 1.º para p em [0.0000, 0.3686]; acima passa F2


  simulação (±1 degrau): F3 em 1.º em 100.0% das repetições

confere com os slides: sim
